In [ ]:
import os
import shutil
import subprocess
import glob

os.chdir("/content")

print("📦 Установка gdown...")
!pip install -q gdown

# =====================================================================
# 🌐 АВТОМАТИЧЕСКАЯ ЗАГРУЗКА ЧАСТЕЙ АРХИВА С GOOGLE DRIVE
# =====================================================================
parts_info = [
    ("1WmL2k-CORm9SkzX6AgN9cOAYuHBwqtf_", "/content/AIGurlDeps.tar.gz.00"),
    ("1sH0av4ikJz2U54kvkzHMkWHQxW-wTZD1", "/content/AIGurlDeps.tar.gz.01"),
    ("1CmTVYpiWUbq6kFyYtdMwesAxSJB2euax", "/content/AIGurlDeps.tar.gz.02")
]

for file_id, dest_path in parts_info:
    if not os.path.exists(dest_path):
        print(f"📥 Скачивание части архива ({os.path.basename(dest_path)})...")
        !gdown --id {file_id} -O {dest_path}

print("📦 Установка системных зависимостей (sox, libsox-dev, pigz, unzip)...")
!apt-get update -y && apt-get install -y sox libsox-dev pigz unzip -qq

print("🚀 Установка Miniconda (Python 3.10) с принятием TOS...")
!export CONDA_PLUGINS_AUTO_ACCEPT_TOS=true && \
wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh && \
bash miniconda.sh -b -p /content/miniconda && \
rm miniconda.sh

print("🚀 Клонирование CosyVoice рекурсивно...")
!git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git

# =====================================================================
# 🌐 СКЛЕЙКА И РАСПАКОВКА ЧАСТЕЙ АРХИВА
# =====================================================================
tar_parts = sorted(glob.glob("/content/AIGurlDeps.tar.gz.*"))
tar_path = "/content/AIGurlDeps.tar.gz"

if tar_parts and not os.path.exists(tar_path):
    print(f"📦 Найдены части архива ({len(tar_parts)} шт.). Склеиваем в {tar_path}...")
    with open(tar_path, "wb") as wfd:
        for f in tar_parts:
            with open(f, "rb") as rfd:
                shutil.copyfileobj(rfd, wfd)
    print("✅ Склейка архива завершена!")

os.chdir("/content/CosyVoice")
!git submodule update --init --recursive

if os.path.exists(tar_path):
    print("📦 Найден полный архив зависимостей AIGurlDeps.tar.gz. Распаковываем...")
    !apt-get update -y && apt-get install -y pigz -qq
    !tar -I pigz -xf AIGurlDeps.tar.gz -C /content
    print("✅ Окружение и CosyVoice успешно восстановлены из архива!")
else:
    print("🛠️ Удаление openai-whisper из requirements.txt для предотвращения сбоя сборки...")
    !sed -i '/openai-whisper/d' requirements.txt

    print("⚙️ Создание conda-окружения 'cosyvoice' и установка requirements.txt...")
    install_cmds = (
        "export PATH=/content/miniconda/bin:$PATH && "
        "conda create -n cosyvoice -y python=3.10 && "
        "conda run -n cosyvoice pip install -r requirements.txt -i https://mirrors.aliyun.com/pypi/simple/ --trusted-host=mirrors.aliyun.com && "
        "conda run -n cosyvoice pip install openai-whisper --prefer-binary"
    )
    subprocess.run(["bash", "-c", install_cmds], check=True)

    print("📥 Скачивание претренированных моделей и ttsfrd (ModelScope SDK)...")
    download_models_cmd = (
        "export PATH=/content/miniconda/bin:$PATH && "
        "conda run -n cosyvoice python -c \""
        "from modelscope import snapshot_download; "
        "import os; "
        "os.makedirs('pretrained_models', exist_ok=True); "
        "snapshot_download('FunAudioLLM/Fun-CosyVoice3-0.5B-2512', local_dir='pretrained_models/Fun-CosyVoice3-0.5B'); "
        "snapshot_download('iic/CosyVoice2-0.5B', local_dir='pretrained_models/CosyVoice2-0.5B'); "
        "snapshot_download('iic/CosyVoice-300M', local_dir='pretrained_models/CosyVoice-300M'); "
        "snapshot_download('iic/CosyVoice-300M-SFT', local_dir='pretrained_models/CosyVoice-300M-SFT'); "
        "snapshot_download('iic/CosyVoice-300M-Instruct', local_dir='pretrained_models/CosyVoice-300M-Instruct'); "
        "snapshot_download('iic/CosyVoice-ttsfrd', local_dir='pretrained_models/CosyVoice-ttsfrd')"
        "\""
    )
    subprocess.run(["bash", "-c", download_models_cmd], check=True)

    print("📦 Распаковка и установка ttsfrd для качественной нормализации текста...")
    ttsfrd_install_cmd = (
        "export PATH=/content/miniconda/bin:$PATH && "
        "cd pretrained_models/CosyVoice-ttsfrd/ && "
        "if [ -f resource.zip ]; then unzip -o resource.zip -d .; fi && "
        "conda run -n cosyvoice pip install ttsfrd_dependency-0.1-py3-none-any.whl && "
        "conda run -n cosyvoice pip install ttsfrd-0.4.2-cp310-cp310-linux_x86_64.whl"
    )
    subprocess.run(["bash", "-c", ttsfrd_install_cmd], check=True)

    other_deps_install_cmd = (
        "export PATH=/content/miniconda/bin:$PATH && "
        "conda run -n cosyvoice pip install --upgrade pip wheel 'setuptools==69.5.1' && "
        "conda run -n cosyvoice pip install matcha-tts"
    )
    subprocess.run(["bash", "-c", other_deps_install_cmd], check=True)

    print("✅ Официальная установка окружения 'cosyvoice', моделей и ttsfrd успешно завершена!")

# Переходим в рабочую директорию CosyVoice
os.chdir("/content/CosyVoice")

# =====================================================================
# 📝 ЗАПИСЬ КОДА СЕРВЕРА В server.py
# =====================================================================
server_code = r'''
import os
import json
import tempfile
import urllib.request
import asyncio
import soundfile as sf
from fastapi import FastAPI, WebSocket, WebSocketDisconnect, Response
import uvicorn
import torch
import torchaudio
import numpy as np
import whisper
from pyngrok import ngrok
import sys
import nest_asyncio

nest_asyncio.apply()

sys.path.append(os.path.abspath("."))
from cosyvoice.cli.cosyvoice import AutoModel

app = FastAPI()

print("🎙️ Инициализация Whisper (small) и CosyVoice 2 (AutoModel) на GPU...")
whisper_model = whisper.load_model("small", device="cuda")

# Используем скачанную локально модель CosyVoice2
cosyvoice = AutoModel(model_dir='pretrained_models/CosyVoice2-0.5B')
print("✅ Whisper и CosyVoice 2 успешно загружены!")

CHARACTERS_CONFIG = {
    "sofia": {
        "excited": {"filename": "excited.mp3", "text": "Слушай, представляешь! Я раньше так сильно увлекалась мемами, что мы с подругой даже свой паблик вели!"},
        "sad": {"filename": "sad.mp3", "text": "Ты знаешь... И на этом все наши хиханьки-хаханьки закончились на длительное время."},
        "ironic": {"filename": "ironic.mp3", "text": "На самом деле я сейчас живу в пристройке у родителей, потому что в своей квартире я делаю ремонт. Ну просто прелесть, а не жизнь."},
        "flirty": {"filename": "flirty.mp3", "text": "Знаешь, я тут подумала..."},
        "surprised": {"filename": "surprised.mp3", "text": "Подожди-ка... Ты хочешь сказать, что мы действительно собрали всю эту коллекцию с нуля и никто даже не заметил?! Это просто невероятно!"},
        "angry": {"filename": "angry.mp3", "text": "Сколько можно повторять одно и то же?! Я уже битый час пытаюсь объяснить элементарные вещи, а вы снова за своё!"},
        "tired": {"filename": "tired.mp3", "text": "Ох... как же я устала сегодня... Весь день этот ремонт, дела, куча забот..."},
        "passionate": {"filename": "passionate.mp3", "text": "Ах... как же жарко сегодня... Подойди ближе ко мне, я хочу сказать тебе кое-что очень тихо... Ты даже не представляешь, как сильно я этого ждала..."},
        "moan_cheerful": {"filename": "moan_cheerful.mp3", "text": "Ох... как же это приятно... Продолжай, не останавливайся... мне так хорошо с тобой..."},
    }
}

GITHUB_RAW_BASE = "https://raw.githubusercontent.com/ChronoMonochrome/AIGurl/refs/heads/master/voices"
LOADED_CHARACTERS = {}

for char_name, refs in CHARACTERS_CONFIG.items():
    char_dir = os.path.join("voices", char_name)
    os.makedirs(char_dir, exist_ok=True)
    LOADED_CHARACTERS[char_name] = {}
    for ref_key, data in refs.items():
        mp3_path = os.path.join(char_dir, data["filename"])
        wav_filename = data["filename"].rsplit(".", 1)[0] + "_16k.wav"
        wav_path = os.path.join(char_dir, wav_filename)

        if not os.path.exists(mp3_path):
            file_url = f"{GITHUB_RAW_BASE}/{char_name}/{data['filename']}"
            try:
                urllib.request.urlretrieve(file_url, mp3_path)
            except Exception as e:
                print(f"⚠️ Ошибка скачивания {file_url}: {e}")

        if os.path.exists(mp3_path) and not os.path.exists(wav_path):
            try:
                speech, sr = torchaudio.load(mp3_path)
                if speech.shape[0] > 1:
                    speech = speech.mean(dim=0, keepdim=True)
                if sr != 16000:
                    resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
                    speech = resampler(speech)
                torchaudio.save(wav_path, speech, 16000)
                print(f"✅ Подготовлен референс (16kHz, mono): {wav_path}")
            except Exception as e:
                print(f"⚠️ Ошибка ресемпла референса {mp3_path}: {e}")

        if os.path.exists(wav_path):
            LOADED_CHARACTERS[char_name][ref_key] = {
                "prompt_speech_16k": wav_path,
                "prompt_text": data["text"]
            }

print("✅ Референсы голосов подготовлены для CosyVoice 2!")

@app.post("/tts")
async def tts_endpoint(data: dict):
    raw_text = data.get("text", "")
    requested_char = data.get("character", "sofia")
    requested_ref = data.get("ref", "excited")
    if not raw_text:
        return {"error": "No text provided"}

    char_refs = LOADED_CHARACTERS.get(requested_char, LOADED_CHARACTERS["sofia"])
    ref_data = char_refs.get(requested_ref, char_refs.get("excited", {}))

    if not ref_data:
        return {"error": "Reference voice not found"}, 400

    try:
        # Вызов zero-shot с отключенным внешним фронтендом для избежания галлюцинаций
        output_generator = cosyvoice.inference_zero_shot(
            raw_text,
            ref_data["prompt_text"],
            ref_data["prompt_speech_16k"],
            stream=False,
            text_frontend=False
        )

        audio_chunks = []
        sr = cosyvoice.sample_rate
        for i, result in enumerate(output_generator):
            if 'tts_speech' in result:
                chunk = result['tts_speech'].detach().cpu().numpy()
                audio_chunks.append(chunk)

        if not audio_chunks:
            return {"error": "TTS generation failed"}, 500

        wav = np.concatenate(audio_chunks, axis=-1)
        if wav.ndim > 1:
            wav = wav.squeeze()

        padding = np.zeros(int(sr * 0.15), dtype=wav.dtype)
        wav = np.concatenate([wav, padding])

        temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
        sf.write(temp_file.name, wav, sr)
        temp_file.close()

        with open(temp_file.name, "rb") as f:
            audio_bytes = f.read()
        os.remove(temp_file.name)
        return Response(content=audio_bytes, media_type="audio/wav")

    except Exception as e:
        print(f"❌ Ошибка синтеза CosyVoice: {e}")
        return {"error": str(e)}, 500

@app.websocket("/ws/voice")
async def voice_websocket(websocket: WebSocket):
    await websocket.accept()
    print("🔌 Клиент подключен по WebSocket (дуплекс)")

    audio_buffer = bytearray()
    transcription_history = []

    try:
        while True:
            try:
                message = await asyncio.wait_for(websocket.receive(), timeout=30.0)
            except asyncio.TimeoutError:
                await websocket.send_json({"type": "ping"})
                continue

            if "bytes" in message:
                audio_buffer.extend(message["bytes"])

            elif "text" in message:
                data = json.loads(message["text"])
                action = data.get("action")

                if action == "speech_end":
                    if len(audio_buffer) > 0:
                        temp_audio = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
                        temp_name = temp_audio.name
                        temp_audio.close()

                        audio_np = np.frombuffer(audio_buffer, dtype=np.int16).astype(np.float32) / 32768.0
                        sf.write(temp_name, audio_np, 16000)
                        audio_buffer.clear()

                        dynamic_prompt = "Разговор Дениса и девушки. " + " ".join(transcription_history[-6:])

                        print("🎙️ Распознавание речи через Whisper...")
                        result = whisper_model.transcribe(
                            temp_name,
                            language="ru",
                            initial_prompt=dynamic_prompt,
                            condition_on_previous_text=False,
                            beam_size=5
                        )
                        user_text = result.get("text", "").strip()

                        if os.path.exists(temp_name):
                            os.remove(temp_name)

                        print(f"📝 Распознанный текст: '{user_text}'")

                        if user_text:
                            transcription_history.append(f"Денис: {user_text}")

                        await websocket.send_json({
                            "type": "transcription",
                            "text": user_text
                        })
                elif action == "assistant_said":
                    bot_text = data.get("text", "")
                    if bot_text:
                        transcription_history.append(f"София: {bot_text}")
                elif action in ("clear_buffer", "interrupt"):
                    audio_buffer.clear()
                    print("⚡ Буфер аудио на сервере очищен.")

    except WebSocketDisconnect:
        print("🔌 Клиент отключился от WebSocket")
    except Exception as e:
        print(f"⚠️ Ошибка в WebSocket: {e}")

if __name__ == "__main__":
    NGROK_TOKEN = ""
    STATIC_URL = ""

    try:
        ngrok.set_auth_token(NGROK_TOKEN)
    except:
        pass

    public_url = ngrok.connect(8000, url=STATIC_URL).public_url
    print(f"\n🌐 Статический туннель ngrok успешно запущен: {public_url}\n")

    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

with open("server.py", "w", encoding="utf-8") as f:
    f.write(server_code)

print("🚀 Запуск сервера через Python 3.10 в режиме реального времени...")
process = subprocess.Popen(
    ["/content/miniconda/envs/cosyvoice/bin/python", "server.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line, end="")

process.wait()
if process.returncode != 0:
    raise subprocess.CalledProcessError(process.returncode, "server.py")

In [ ]:
from google.colab import drive
import glob
import shutil
import os

# 1. Монтируем Google Drive
print("🔗 Подключение Google Drive...")
drive.mount('/content/drive')

# 2. Создаем папку для бэкапа на Драйве
backup_dir = '/content/drive/MyDrive/AIGurl_Backup'
os.makedirs(backup_dir, exist_ok=True)

# 3. Находим все части архива (AIGurlDeps.tar.gz.00, .01 и т.д.)
archive_parts = sorted(glob.glob('/content/AIGurlDeps.tar.gz.*'))

if not archive_parts:
    print("⚠️ Части архива не найдены в директории /content/")
else:
    print(10 * "-" + f"\nНайдено частей для копирования: {len(archive_parts)}")
    for part in archive_parts:
        filename = os.path.basename(part)
        dest_path = os.path.join(backup_dir, filename)

        print(f"📦 Копирование {filename} на Google Drive...")
        shutil.copy(part, dest_path)
        print(f"✅ Успешно скопировано в: {dest_path}")

    print("\n🎉 Все части архива надежно сохранены на вашем Google Drive!")
    print(f"📁 Путь на диске: Драйв -> AIGurl_Backup")